<div style="
  background-color:#ffffff;
  color:#1f2937;
  padding:28px 24px;
  border-radius:14px;
  border:1px solid #d1d5db;
  font-family:'Segoe UI', Arial, sans-serif;
  text-align:center;
  box-shadow:0 6px 18px rgba(0,0,0,0.06);
">

  <div style="text-align:center; margin-bottom:18px;">
    <img src="../Su.svg.png" alt="Sorbonne Université Logo" style="
      width:105px;
      height:auto;
      display:block;
      margin:0 auto;
    ">
  </div>

  <div style="
    display:inline-block;
    font-size:22px;
    font-weight:700;
    color:#111827;
    padding-bottom:8px;
    margin-bottom:16px;
  ">
    A-Abo
  </div>

  <div style="font-size:18px; font-weight:600; color:#374151;">
    Projet académique de bases de données
  </div>

  <div style="font-size:15px; margin-top:6px; color:#1e3a8a; font-weight:600;">
    Sorbonne Université — Année universitaire 2025–2026
  </div>

  <div style="
    margin:18px auto 0 auto;
    max-width:650px;
    font-size:12px;
    line-height:1.5;
    color:#6b7280;
    font-style:italic;
  ">
    Ce document est fourni à titre pédagogique et peut faire l’objet de corrections ou d’améliorations.
  </div>

</div>

<center><h2> TD 6 et TME6 : REQUÊTES IMBRIQUÉES AVEC EXISTS, ALL ET ANY

<center></center> <h5> Ce TD/TME utilise les données contenues dans le fichier --bd_jo_v2_H2.sql--

# Installation H2 

Le système utilisé pendant les TME est H2.

Télécharger le pilote de H2

In [1]:
pip install psycopg2-binary

Note: you may need to restart the kernel to use updated packages.


https://www.psycopg.org/docs/
http://localhost:8082

**Relancez le kernel**: Kernel -> Restart kernel ...

In [2]:
import psycopg2
import os
local_dir = "./data"
os.makedirs(local_dir, exist_ok=True)
os.listdir(local_dir)

[]

**Attention**: vous pouvez cliquer sur les 3 lignes dans la fenêtre de gauche pour d'afficher les différentes sections du notebook 

# Démarrage du serveur H2
Démarrer un serveur de base de données H2 en arrière-plan, accessible uniquement localement(localhost, 127.0.0.1) sur le port 8082, avec les fichiers des bases de données stockés dans ./data (un fichier *.mv.db par base de données)

In [3]:
port = 8082
print(f'Le numero du port utilisé pour la connexion à la BD est: {port}')

Le numero du port utilisé pour la connexion à la BD est: 8082


In [4]:
%%bash --bg --out output -s "$port"
java -Dh2.bindAddress=127.0.0.1 -cp h2-2.1.214.jar:postgresql-42.6.0.jar org.h2.tools.Server -pg -pgPort $1 -baseDir ./data -ifNotExists

Couldn't find program: 'bash'


Tester si le serveur a été démarré

In [5]:
%%bash
nc -zv 127.0.0.1 8082

Couldn't find program: 'bash'


<span style="color:red"> 
<center> <h1>ADD THIS PART FOR WINDOWS !!

In [162]:
import psycopg2
import os
import socket
import subprocess
import time

In [163]:
local_dir = "./data"
os.makedirs(local_dir, exist_ok=True)
os.listdir(local_dir)

['jo8082.mv.db']

In [164]:
port = 8082
print(f"Le numero du port utilisé pour la connexion à la BD est: {port}")

Le numero du port utilisé pour la connexion à la BD est: 8082


In [165]:
cmd = [
    "java",
    "-Dh2.bindAddress=127.0.0.1",
    "-cp",
    "h2-2.1.214.jar;postgresql-42.6.0.jar",
    "org.h2.tools.Server",
    "-pg",
    "-pgPort",
    str(port),
    "-baseDir",
    "./data",
    "-ifNotExists"
]

h2_process = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True
)

time.sleep(2)

if h2_process.poll() is None:
    print("H2 server started successfully in background.")
else:
    out, err = h2_process.communicate()
    print("H2 server failed to start.")
   # print("STDOUT:\n", out)
    #print("STDERR:\n", err)
    # cause Already running check connection = connect_H2( .... ) execute the function H2 and the line after it 

H2 server failed to start.


In [166]:
s = socket.socket()
result = s.connect_ex(("127.0.0.1", port))
s.close()

if result == 0:
    print(f"Server is running on port {port}")
else:
    print(f"Server is NOT running on port {port}")

Server is running on port 8082


# Connexion du client au serveur H2
Se connecter au serveur H2 précédemment lancé en utilisant un nom de base de données, un login et un mot de passe. Si une connexion existante existe, elle est fermée avant d’en créer une nouvelle. Si la base n’existe pas encore, H2 la crée automatiquement dans le répertoire de données ./data (fichier *.mv.db). La fonction retourne un objet de connexion utilisable pour exécuter des commandes SQL sur le serveur.

In [167]:
def connect_H2(db,user,password):
    global connection
    try:
        connection
    except:
        connection = None
    if connection != None:
        try:
            connection.close()
            print("Connection closed")
        except  Error as e:
            print(f"The error '{e}' occurred")
    try:
        connection = psycopg2.connect(f"dbname={db} user={user} password={password} host=localhost port={port}")
        print("Connection to H2 DB successful")
    except Exception as e:
        print(f"The error '{e}' occurred")
    return connection

In [168]:
#Connexion à la base H2 nommée "jo<port>" avec l’utilisateur et le mot de passe spécifiés. Si la base n’existe pas encore, H2 la crée automatiquement. 
#Toute connexion précédente est fermée avant d’en établir une nouvelle.
connection = connect_H2("jo"+f'{port}',"user","pass")

Connection closed
Connection to H2 DB successful


# Fonction execute

Exécute une commande SQL sur la connexion fournie, affiche le nombre de lignes affectées et, si demandé, présente les résultats sous forme de tableau lisible avec les noms de colonnes. Le curseur peut être fermé automatiquement après l’exécution, et toute erreur est affichée. La fonction retourne le curseur utilisé pour accéder aux résultats ou None en cas d’échec.

In [169]:
def execute(connection, query, show=True, close=True, top=10):
    try:
        cursor = connection.cursor()
        cursor.execute(query)
        if cursor.rowcount >0:
          print(cursor.rowcount,"rows,", f"display only {top} rows")
        else:
          print(cursor.rowcount,"rows")
        if show and cursor.rowcount:
            names = [desc[0] for desc in cursor.description]

            lengths={}
            for attr in names:
                lengths[attr]=len(attr)
            for ligne in cursor:
                i=0
                for attr in ligne:
                    lengths[names[i]]=max(lengths[names[i]],len(str(attr)))
                    i=i+1
            print('|',end='')
            for attr in names:
                print(str(attr).ljust(lengths[attr]),end='|')
            print()
            print('|',end='')
            for attr in names:
                print(''.ljust(lengths[attr]+1,'-'),end='')
            print()
            cursor.execute(query)
            n=0
            for ligne in cursor:
                i=0
                print('|',end='')
                for attr in ligne:
                    print(str(attr)[:lengths[names[i]]].ljust(lengths[names[i]]),end='|')
                    i+=1
                print()
                n+=1
                if n>top :
                  break
        if close:
            cursor.close()
    except Exception as e:
        print(f"The error '{e}' occurred")
        cursor=None
    return cursor

# Affichage du schéma d'une table

In [170]:
def show_table(connection,table_name):

    print(f'*********** Attributs de la table : {table_name} ************')
    query=f"""
        SELECT table_name,
              column_name,
              data_type,
              column_default,
              is_nullable,
              character_maximum_length,
              numeric_precision
        FROM information_schema.columns
        where lower(table_name)  = '{table_name}'
        """
    execute(connection,query)


    print(f'\n*********** Contraintes de la table : {table_name} ************')
    query = f"""
        SELECT tc.constraint_name, 
               tc.constraint_type, 
               cc.check_clause
        FROM information_schema.table_constraints tc
        LEFT JOIN information_schema.check_constraints cc 
               ON tc.constraint_name = cc.constraint_name
        WHERE lower(tc.table_name) = '{table_name}'
        AND tc.table_schema = 'public'
        ORDER BY tc.constraint_type
        """
    execute(connection, query)

# Affichage du schéma global 

In [171]:
def show_schema(connection):

    print('*********** Tables ************')
    query="""
    select TABLE_NAME
    from INFORMATION_SCHEMA.TABLES
    where TABLE_SCHEMA = 'public'
    """
    execute(connection,query,show=True,top=1000)

    print('\n\n*********** Domaines ************')
    query="""
    SELECT domain_name,check_clause
    FROM information_schema.domain_constraints a left outer join
         information_schema.check_constraints b
         on a.constraint_name=b.constraint_name
    """
    execute(connection,query,top=1000)

    print('\n\n*********** Attributs ************')
    query=f"""
    SELECT c.table_name,
           c.column_name,
           c.data_type,
           c.column_default,
           c.is_nullable,
           c.character_maximum_length,
           c.numeric_precision
    FROM INFORMATION_SCHEMA.TABLES t, information_schema.columns c
    where t.table_name=c.table_name
    and t.TABLE_SCHEMA = 'public'
    """
    execute(connection,query,top=1000)

    print('\n\n*********** Contraintes d''Intégrité ************')
    query="""
    SELECT 
        tc.table_name, 
        tc.constraint_name, 
        tc.constraint_type,
        cc.check_clause
    FROM information_schema.table_constraints tc
    LEFT JOIN information_schema.check_constraints cc 
        ON tc.constraint_name = cc.constraint_name
    WHERE tc.table_schema = 'public'
    ORDER BY tc.table_name, tc.constraint_type
    """
    execute(connection,query,top=1000)

# TD6

On considère le schéma de la base JEUXOLYMPIQUE2014 qui décrit les athlètes et leurs résultats aux
épreuves des sports des Jeux Olympiques d'Hiver Sotchi 2014 :

**PAYS** ( <u>CODEPAYS</u>, NOMP)<br/>
Ex. ('FRA', 'France')<br/>
**SPORT** ( <u>SID</u>, NOMSP)<br/>
Ex. (1, 'Biathlon')<br/>
**EPREUVE** ( <u>EPID</u>, SID*, NOMEP, CATÉGORIE, DATEDEBUT, DATEFIN)<br/>
Ex. (10, 1, 'relais 4x7,5km', 'Hommes', 22/02/2014, 22/02/2014)<br/>
**ATHLETE** ( <u>AID</u>, NOMATH, PRENOMATH, DATENAISSANCE, CODEPAYS*)<br/>
Ex. (1000, 'SOBOLEV', 'Alexey', NULL, 'RUS')<br/>
**EQUIPE** ( <u>EQID</u>, CODEPAYS*)<br/>
Ex. (30, 'SUI')<br/>
**ATHLETESEQUIPE** ( <u>EQID*, AID*</u>)<br/>
Ex. (30, 796) : L'athlète (aid=796) a participé à l'équipe (eqid=30)<br/>
**RANGINDIVIDUEL** ( <u>EPID*, AID*</u>, RANG)<br/>
Ex. (15, 61, 1) : L'athlète (aid=61) a gagné la médaille d'or (rang=1) de l'épreuve (epid=15)<br/>
**RANGEQUIPE** ( <u>EPID*, EQID*</u>, RANG)<br/>
Ex. (10, 30, 14) : L'équipe (eqid=30) a été classée 14e à l'épreuve (epid=10)<br/>


Les attributs qui forment la clé primaire de chaque relation sont soulignés. Les clés étrangères sont
signalées avec une *. Les attributs aid, epid, eqid et sid correspondent aux identifiants des athlètes,
épreuves, équipes et sports et sont utilisés à la fois comme clé primaire ou comme référence (clé
étrangère) vers la relation correspondante.
La relation **PAYS** contient le code et le nom de tous les pays, même si ils n'ont pas participé aux
Jeux Olympiques. Les sports (n-uplets de la relation **SPORT**) sont un ensemble d'épreuves (n-uplets
de la relation **EPREUVE**). Pour chaque épreuve on connaît son nom et les date de début et fin de
l'épreuve. Les épreuves peuvent être individuelles ou par équipe. Dans le premier cas, la
participation des athlètes (n-uplets de la relation ATHLETE) est stocké dans la table
**RANGINDIVIDUEL** qui contient en plus le rang qu'ils ont obtenu (1 pour la médaille d'or). Pour les
épreuves par équipe les résultats sont stockés dans la relation **RANGEQUIPE**, alors que l'information
sur le pays de chaque équipe et ses participants et stocké dans les relations **EQUIPE** et
**ATHLETESEQUIPE**. Dans les relations **RANGINDIVIDUEL** et **RANGEQUIPE** l'attribut rang est égal à
null si l'athlète ou l'équipe a été disqualifié.




## Créer les tables et charger les données bd_jo_v2_H2.sql

In [172]:
schemafile=open("bd-jo-v2_H2.sql",mode="r",encoding='utf-8')
create_schema=schemafile.read()
execute(connection,"drop all objects")
execute(connection, create_schema)
connection.commit()

0 rows
0 rows


**Requêtes: ANY /ALL / IN / EXISTS**

## **Q1**. 
Les athlètes ayant participé à (au moins) une épreuve individuelle et (au moins) une épreuve par équipe. Exprimer la requête en utilisant ANY (au lieu de IN; voir requête 2.6 dans TD5).
Résultat : (372 lignes)

In [17]:
query="""
select a.*
from athlete a
where a.aid = any(select r.aid
                    from rangindividuel r )
and a.aid = any (select aq.aid 
                from athletesequipe aq)
"""
execute(connection,query,show=True)

372 rows, display only 10 rows
|aid |nomath             |prenomath        |datenaissance|codepays|
|------------------------------------------------------------------
|1   |BJOERNDALEN        |Ole Einar        |1974-01-27   |NOR     |
|2   |BJOERGEN           |Marit            |1980-03-21   |NOR     |
|3   |AN                 |Victor           |1985-11-23   |RUS     |
|5   |WÜST               |Ireen            |1986-04-01   |NED     |
|6   |SVENDSEN           |Emil Hegle       |1985-07-12   |NOR     |
|8   |KRAMER             |Sven             |1986-04-23   |NED     |
|11  |MORGENSTERN        |Thomas           |1986-10-30   |AUT     |
|12  |HELLNER            |Marcus           |1985-11-25   |SWE     |
|14  |DOMRACHEVA         |Darya            |1986-08-03   |BLR     |
|15  |COLOGNA            |Dario            |1986-03-11   |SUI     |
|16  |LOCH               |Felix            |1989-07-24   |GER     |


<cursor object at 0x00000193FD289EE0; closed: -1>

 ## **Q2. a)** 
Les pays ayant au moins un athlète (requête équivalente: les pays ayant participé aux JO).
Résultat : (88 lignes)

In [18]:
query="""
select p.*
from pays p
where p.codepays = any (select a.codepays 
                        from athlete a)
"""
execute(connection,query,show=True)

88 rows, display only 10 rows
|codepays|nomp                                 |
|-----------------------------------------------
|ALB     |Albanie                              |
|AND     |Andorre                              |
|ARG     |Argentine                            |
|ARM     |Arménie                              |
|AUS     |Australie                            |
|AUT     |Autriche                             |
|AZE     |Azerbaïdjan                          |
|BEL     |Belgique                             |
|BER     |Bermudes                             |
|BIH     |Bosnie-Herzégovine                   |
|BLR     |Biélorussie                          |


<cursor object at 0x00000193FD28A0A0; closed: -1>

##  **Q2. b**) 
Les pays ayant exactement un seul athlète
Résultat : (18 lignes)

In [19]:
query="""
select distinct p.nomp
from pays p , athlete a 
where p.codepays =a.codepays 
and not exists (select *
                from athlete a2 
                where a2.aid != a.aid and a2.codepays = p.codepays)
"""
execute(connection,query,show=True)

18 rows, display only 10 rows
|nomp                       |
|----------------------------
|Bermudes                   |
|Hong Kong                  |
|Kirghizistan               |
|Luxembourg                 |
|Malte                      |
|Mexique                    |
|Népal                      |
|Pakistan                   |
|Paraguay                   |
|Philippines                |
|Tadjikistan                |


<cursor object at 0x00000193FD289C40; closed: -1>

## **Q3. a)** 
Les athlètes qui n'ont jamais été disqualifiés aux épreuves individuelles.
Résultat : (2194 lignes)

In [20]:
query="""
select a.*
from athlete a
where a.aid != all (select r.aid 
                    from rangindividuel r
                    where r.rang is null)
"""
# not exist ( ... )
# not in ( .... )
execute(connection,query,show=True)

2194 rows, display only 10 rows
|aid |nomath                 |prenomath        |datenaissance|codepays|
|----------------------------------------------------------------------
|1   |BJOERNDALEN            |Ole Einar        |1974-01-27   |NOR     |
|2   |BJOERGEN               |Marit            |1980-03-21   |NOR     |
|3   |AN                     |Victor           |1985-11-23   |RUS     |
|4   |PECHSTEIN              |Claudia          |1972-02-22   |GER     |
|5   |WÜST                   |Ireen            |1986-04-01   |NED     |
|6   |SVENDSEN               |Emil Hegle       |1985-07-12   |NOR     |
|7   |AMMANN                 |Simon            |1981-06-25   |SUI     |
|8   |KRAMER                 |Sven             |1986-04-23   |NED     |
|9   |SABLIKOVA              |Martina          |1987-05-27   |CZE     |
|10  |HAMELIN                |Charles          |1984-04-14   |CAN     |
|11  |MORGENSTERN            |Thomas           |1986-10-30   |AUT     |


<cursor object at 0x00000193FD28A260; closed: -1>

## **Q3. b)** 
Les pays qui n'ont pas eu d'athlète disqualifié aux épreuves individuelles.
Résultat : (143 lignes)

In [21]:
query="""
select p.*
from pays p
where p.codepays != all (select a.codepays
                        from athlete a , rangindividuel r
                        where r.aid = a.aid and  r.rang is null)
"""
# not exist ( .... )
# not in ( .... )
execute(connection,query,show=True)

143 rows, display only 10 rows
|codepays|nomp                            |
|------------------------------------------
|AFG     |Afghanistan                     |
|ALG     |Algérie                         |
|ANG     |Angola                          |
|ANT     |Antigua-et-Barbuda              |
|ARU     |Aruba                           |
|ASA     |Samoa américaines               |
|BAH     |Bahamas                         |
|BAN     |Bangladesh                      |
|BAR     |Barbade                         |
|BDI     |Burundi                         |
|BEL     |Belgique                        |


<cursor object at 0x00000193FD2897E0; closed: -1>

##  **Q3. c)** 
Les pays qui ont participé aux JO et qui n'ont pas eu d'athlètes disqualifiés aux épreuves
individuelles.
Résultat : (25 lignes)

In [22]:
query="""
select distinct p.*
from pays p 
where exists (select *
                from athlete a
                where  p.codepays  = a.codepays  )
and not exists (select *
                from athlete a2 ,rangindividuel r2
                where a2.aid = r2.aid and a2.codepays= p.codepays and r2.rang is null)
"""
# not in ( ... )
# != all ( ... )
execute(connection,query,show=True)

25 rows, display only 10 rows
|codepays|nomp                       |
|-------------------------------------
|BEL     |Belgique                   |
|BER     |Bermudes                   |
|DEN     |Danemark                   |
|EST     |Estonie                    |
|HKG     |Hong Kong                  |
|IND     |Inde                       |
|IRI     |Iran                       |
|ISV     |Îles Vierges des États-Unis|
|IVB     |Îles Vierges britanniques  |
|JAM     |Jamaïque                   |
|KAZ     |Kazakhstan                 |


<cursor object at 0x00000193FD28A340; closed: -1>

##  **Q3. d)** 
Les pays qui ont participé aux JO et qui n'ont pas eu des athlètes disqualifiés ni en
individuel ni par équipe
Résultat : (25 lignes)

In [26]:
query="""
select distinct p.*
from pays p 
where exists (select *
                from athlete a
                where  p.codepays  = a.codepays )
and not exists (select *
                from athlete a2 ,rangindividuel r2
                where a2.aid = r2.aid and a2.codepays= p.codepays and r2.rang is null)
and not exists (select *
                from athlete a3, athletesequipe aq , rangequipe rq
                where a3.codepays = p.codepays and aq.aid = a3.aid and rq.eqid = aq.eqid   and   rq.rang is null)

"""
execute(connection,query,show=True)

25 rows, display only 10 rows
|codepays|nomp                       |
|-------------------------------------
|BEL     |Belgique                   |
|BER     |Bermudes                   |
|DEN     |Danemark                   |
|EST     |Estonie                    |
|HKG     |Hong Kong                  |
|IND     |Inde                       |
|IRI     |Iran                       |
|ISV     |Îles Vierges des États-Unis|
|IVB     |Îles Vierges britanniques  |
|JAM     |Jamaïque                   |
|KAZ     |Kazakhstan                 |


<cursor object at 0x00000193FD28AB20; closed: -1>

##  **Q4. a)** 
Les athlètes n'ayant pas gagné de médaille ni en individuel ni en équipe
Résultat : (1921 lignes)

In [28]:
query="""
select a.*
from athlete a
where not exists (select *
                  from rangindividuel ri
                   where a.aid = ri.aid and ri.rang <= 3)
and not exists (select *
                from rangequipe rq , athletesequipe aq
                where rq.eqid = aq.eqid and aq.aid = a.aid and rq.rang <= 3)
"""
execute(connection,query,show=True)

1921 rows, display only 10 rows
|aid |nomath                 |prenomath        |datenaissance|codepays|
|----------------------------------------------------------------------
|4   |PECHSTEIN              |Claudia          |1972-02-22   |GER     |
|7   |AMMANN                 |Simon            |1981-06-25   |SUI     |
|19  |SACHENBACHER-STEHLE    |Evi              |1980-11-27   |GER     |
|22  |PLYUSHCHENKO           |Evgeny           |1982-11-03   |RUS     |
|24  |DAVIS                  |Shani            |1982-08-13   |USA     |
|29  |HENKEL                 |Andrea           |1977-12-10   |GER     |
|31  |NORTHUG                |Petter Jr.       |1986-01-06   |NOR     |
|32  |DI CENTA               |Giorgio          |1972-07-10   |ITA     |
|33  |KUZMINA                |Anastazia        |1984-08-28   |SVK     |
|36  |RAICH                  |Benjamin         |1978-02-28   |AUT     |
|40  |SCHOCH                 |Philipp          |1979-10-12   |SUI     |


<cursor object at 0x00000193FD28B140; closed: -1>

##  **Q4. b)** 
Les pays ayant participé aux JO et n'ayant pas gagné de médaille ni en individuel ni en
équipe.
Résultat : (63 lignes)

In [43]:
query="""
select p.*
from pays p 

where exists (select *
                from athlete a2
                where a2.codepays = p.codepays)
and not exists (select *
                    from athlete a, rangindividuel ri 
                    where a.aid =ri.aid and a.codepays = p.codepays and ri.rang <= 3)
and not exists (select *
                from equipe e , rangequipe rq
                where e.codepays = p.codepays and e.eqid = rq.eqid and rq.rang <= 3 )
"""
execute(connection,query,show=True)

63 rows, display only 10 rows
|codepays|nomp                                 |
|-----------------------------------------------
|ALB     |Albanie                              |
|AND     |Andorre                              |
|ARG     |Argentine                            |
|ARM     |Arménie                              |
|AZE     |Azerbaïdjan                          |
|BEL     |Belgique                             |
|BER     |Bermudes                             |
|BIH     |Bosnie-Herzégovine                   |
|BRA     |Brésil                               |
|BUL     |Bulgarie                             |
|CAY     |Îles Caïmans                         |


<cursor object at 0x00000193FD28BBC0; closed: -1>

##  **Q5.** 
L'(es) épreuve(s) avec la plus grande durée.
Résultat : Hockey sur glace, Hockey sur glace, Hommes, 16 jours

In [182]:
query="""
select distinct e.*
from epreuve e 
where not exists (select *
                    from epreuve e2 
                    where e2.epid <> e.epid and  datediff('day',e2.datedebut,e2.datefin) > datediff('day',e.datedebut,e.datefin))
"""
execute(connection,query,show=True)

1 rows, display only 10 rows
|epid|sid|nomep           |categorie|datedebut |datefin   |
|----------------------------------------------------------
|21  |5  |hockey sur glace|Hommes   |2014-02-08|2014-02-23|


<cursor object at 0x00000193FD8E7D80; closed: -1>

##  **Q6.**  
Les athlètes ayant gagné une médaille à toutes les épreuve individuelles auxquelles ils ont participé. Résultat : (109 lignes)

In [88]:
query="""
select distinct a.*
from athlete a
where exists (select *
                from rangindividuel r2
                where r2.aid  =a.aid) 
and not exists (select * 
                  from rangindividuel r
                  where r.aid = a.aid and (r.rang > 3 or r.rang is null))
"""
execute(connection,query,show=True)

109 rows, display only 10 rows
|aid |nomath         |prenomath       |datenaissance|codepays|
|-------------------------------------------------------------
|3   |AN             |Victor          |1985-11-23   |RUS     |
|5   |WÜST           |Ireen           |1986-04-01   |NED     |
|8   |KRAMER         |Sven            |1986-04-23   |NED     |
|9   |SABLIKOVA      |Martina         |1987-05-27   |CZE     |
|10  |HAMELIN        |Charles         |1984-04-14   |CAN     |
|14  |DOMRACHEVA     |Darya           |1986-08-03   |BLR     |
|16  |LOCH           |Felix           |1989-07-24   |GER     |
|17  |ZHOU           |Yang            |1991-06-09   |CHN     |
|27  |ZOEGGELER      |Armin           |1974-01-04   |ITA     |
|34  |PARK           |Seung-Hi        |1992-03-28   |KOR     |
|39  |GEISENBERGER   |Natalie         |1988-02-05   |GER     |


<cursor object at 0x00000193FD6A5380; closed: -1>

**<u>BASE FOOFLE</u>**

## Créer les tables et charger les données base_foofle_H2.sql

In [89]:
schemafile=open("base_foofle_H2.sql",mode="r",encoding='utf-8')
create_schema=schemafile.read()
execute(connection,"drop all objects")
execute(connection, create_schema)
connection.commit()

0 rows
0 rows


##  **Q7.**  
Quelles équipes ont déjà joué au stade préféré de l'équipe des Piepla ? Résultats : (2 lignes) Direkt , Piepla

In [90]:
query="""
select e.*
from equipe e 
where exists (select *
              from  equipe e2 , match m 
              where m.eq1 = e.neq and e2.neq = 'Piepla' and m.st = e2.stp)
"""
execute(connection,query,show=True)

2 rows, display only 10 rows
|neq   |ville|couleur|stp       |
|--------------------------------
|Piepla|Paris|Rouge  |GrandArena|
|Direkt|Nancy|Bleu   |BukHall   |


<cursor object at 0x00000193FD28A880; closed: -1>

##  **Q8.**  
Quels sont les stades où a déjà joué Manon Messi ? Résultats: (3 lignes) GrandArena, Boulodrome, BukHall

In [91]:
query="""
select m.*
from match m 
where exists (select *
              from joueur j
              where j.eq = m.eq1 and j.njo = 'Manon Messi')
"""
execute(connection,query,show=True)

3 rows, display only 10 rows
|eq1   |eq2      |datem     |st        |
|---------------------------------------
|Direkt|Piepla   |2015-05-12|GrandArena|
|Direkt|Fortiches|2015-05-13|Boulodrome|
|Direkt|Sabar    |2015-06-15|BukHall   |


<cursor object at 0x00000193FD6A65E0; closed: -1>

##  **Q9.**  
A quelle date a eu lieu un match entre deux équipes sponsorisées par le même sponsor ? Résultats :(4 lignes) 2015-05-16, 2015-05-15, 2015-06-15, 2015-05-12

In [101]:
query="""
select distinct m.datem
from match m 
where exists (select *
               from joueur j1, joueur j2 , sponsorise s1 , sponsorise s2
               where j1.eq = m.eq1
               and j2.eq = m.eq2
               and j1.njo <> j2.njo
               and s1.njo = j1.njo
               and s2.njo = j2.njo
               and s1.nsp = s2.nsp
               
)
"""
# IMP  j2.njo <> j1.njo
execute(connection,query,show=True)

4 rows, display only 10 rows
|datem     |
|-----------
|2015-05-12|
|2015-05-15|
|2015-05-16|
|2015-06-15|


<cursor object at 0x00000193FD817220; closed: -1>

##  **Q10.**  
Quel sponsor a financé deux joueurs différents ayant eu un match le même jour et dans des
stades différents mais proches (moins de 50 km) ? Résultats :(4 lignes) Air Monaco, Carouf, Robek, Adadis

In [114]:
query="""
select distinct s.nsp
from sponsorise s
where exists (select *
                from match m1 , match m2 , joueur j1 ,joueur j2 , distance d, sponsorise s1, sponsorise s2
                where m1.eq1 = j1.eq 
                and m2.eq1 = j2.eq 
                and m1.st <> m2.st 
                and j1.njo <>j2.njo 
                and s1.njo = j1.njo
                and s2.njo = j2.njo 
                and s1.nsp = s.nsp
                and s2.nsp = s.nsp
                and m1.datem = m2.datem
                and d.st1 = m1.st
                and d.st2 = m2.st 
                and d.nbkm < 50
                )
"""
execute(connection,query,show=True)

4 rows, display only 10 rows
|nsp       |
|-----------
|Adadis    |
|Air Monaco|
|Carouf    |
|Robek     |


<cursor object at 0x00000193FD816F80; closed: -1>

# TME6

On considère le schéma de la base JEUXOLYMPIQUE2014 donné en TD :

**PAYS** ( <u>CODEPAYS</u>, NOMP)<br/>
Ex. ('FRA', 'France')<br/>
**SPORT** ( <u>SID</u>, NOMSP)<br/>
Ex. (1, 'Biathlon')<br/>
**EPREUVE** ( <u>EPID</u>, SID*, NOMEP, CATÉGORIE, DATEDEBUT, DATEFIN)<br/>
Ex. (10, 1, 'relais 4x7,5km', 'Hommes', 22/02/2014, 22/02/2014)<br/>
**ATHLETE** ( <u>AID</u>, NOMATH, PRENOMATH, DATENAISSANCE, CODEPAYS*)<br/>
Ex. (1000, 'SOBOLEV', 'Alexey', NULL, 'RUS')<br/>
**EQUIPE** ( <u>EQID</u>, CODEPAYS*)<br/>
Ex. (30, 'SUI')<br/>
**ATHLETESEQUIPE** ( <u>EQID*, AID*</u>)<br/>
Ex. (30, 796) : L'athlète (aid=796) a participé à l'équipe (eqid=30)<br/>
**RANGINDIVIDUEL** ( <u>EPID*, AID*</u>, RANG)<br/>
Ex. (15, 61, 1) : L'athlète (aid=61) a gagné la médaille d'or (rang=1) de l'épreuve (epid=15)<br/>
**RANGEQUIPE** ( <u>EPID*, EQID*</u>, RANG)<br/>
Ex. (10, 30, 14) : L'équipe (eqid=30) a été classée 14e à l'épreuve (epid=10)<br/>

**Requêtes: ANY /ALL / IN / EXISTS**

## **Q1.** 
Les sports dont toutes les épreuves ont duré un seul jour.
Résultat : Ski de fond, Ski alpin, Biathlon

In [128]:
query="""
select distinct s.nomsp
from sport s
where not exists (select *
             from epreuve e
             where e.sid = s.sid and datediff('day',e.datedebut,e.datefin) <> 0 ) 
"""
execute(connection,query,show=True)

3 rows, display only 10 rows
|nomsp      |
|------------
|Biathlon   |
|Ski alpin  |
|Ski de fond|


<cursor object at 0x00000193FD8E5460; closed: -1>

## **Q2.** 
Les sports qui n'ont pas d'épreuves de categorie Mixte.
Résultat : (12 lignes)

In [129]:
query="""
select s.nomsp
from sport s
where not exists (select *
                    from epreuve e
                    where e.sid = s.sid and e.categorie ='Mixte')
"""
execute(connection,query,show=True)

12 rows, display only 10 rows
|nomsp                               |
|-------------------------------------
|Bobsleigh                           |
|Combiné nordique                    |
|Curling                             |
|Hockey sur glace                    |
|Patinage de vitesse                 |
|Patinage de vitesse sur piste courte|
|Saut à ski                          |
|Skeleton                            |
|Ski acrobatique                     |
|Ski alpin                           |
|Ski de fond                         |


<cursor object at 0x00000193FD8E42E0; closed: -1>

## **Q3.** 
Les équipes dont aucun athlète n'a gagné de médaille aux épreuves individuelles.
Attention : il y a des équipes sans athlètes.
Résultat : (265 lignes avec les équipes sans athlètes - 252 lignes sans les équipes sans
athlètes)

Avec les équipes sans athlètes :

In [132]:
query="""
select e.eqid 
from equipe e
where not exists (select *
                from rangindividuel ri, athletesequipe aq, athlete a
                where ri.aid = a.aid  and aq.eqid = e.eqid  and aq.aid =a.aid  and ri.rang <=3 )
"""
execute(connection,query,show=True)

265 rows, display only 10 rows
|eqid|
|-----
|65  |
|96  |
|122 |
|221 |
|44  |
|66  |
|92  |
|121 |
|133 |
|169 |
|186 |


<cursor object at 0x00000193FD8E5620; closed: -1>

Sans les équipes sans athlètes :

In [134]:
query="""
select e.eqid 
from equipe e 
where exists (select *
                from athletesequipe aq
                where aq.eqid = e.eqid and aq.aid is not null)
and not exists (select *
                from rangindividuel ri, athletesequipe aq, athlete a
                where ri.aid = a.aid  and aq.eqid = e.eqid  and aq.aid =a.aid  and ri.rang <=3 )
"""
execute(connection,query,show=True)

252 rows, display only 10 rows
|eqid|
|-----
|65  |
|96  |
|122 |
|221 |
|44  |
|66  |
|92  |
|121 |
|133 |
|169 |
|186 |


<cursor object at 0x00000193FD8E5380; closed: -1>

## **Q4.** 
La nationalité de l'athlète le/la plus jeune. Attention : il y a des athlètes dont on ne connaît pas la date de naissance.
Résultat : ('29/11/1998','JPN')

In [145]:
query="""
select a.datenaissance , a.codepays
from athlete a 
where a.datenaissance is not null 
and not exists (select *
                    from athlete a2
                    where  a.datenaissance < a2.datenaissance )
"""
execute(connection,query,show=True)

1 rows, display only 10 rows
|datenaissance|codepays|
|-----------------------
|1998-11-29   |JPN     |


<cursor object at 0x00000193FD8E6340; closed: -1>

## **Q5.** 
Le plus jeune athlète de chaque pays.
Résultat : (26 lignes)

In [159]:
query="""
select distinct a.datenaissance
from pays p , athlete a
where a.codepays = p.codepays and a.datenaissance is not null
and not exists (select *
                from athlete a2 
                where a2.codepays  = p.codepays and a2.aid <> a.aid   and a2.datenaissance > a.datenaissance)
"""
execute(connection,query,show=True)

26 rows, display only 10 rows
|datenaissance|
|--------------
|1979-11-23   |
|1984-08-28   |
|1986-08-03   |
|1986-12-27   |
|1988-10-31   |
|1989-10-05   |
|1990-08-21   |
|1990-09-19   |
|1991-04-27   |
|1991-12-09   |
|1992-01-12   |


<cursor object at 0x00000193FD8E74C0; closed: -1>

# Fermer la connexion

In [183]:
connection.commit() # implicit avec close
connection.close()

In [ ]:
connection